# GENERATE AB-GRID REPORT

Author: Dr. Pierpaolo Calanna, Phd

## 1. IMPORTS

In [2]:
# imports
import re
import json
import yaml
import string
import jinja2 as jn

from pathlib import Path
from cerberus import Validator
from weasyprint import HTML

## 2. CONSTANTS

In [3]:
# folder paths
TEMPLATES_PATH = Path("./templates/")
DATA_PATH = Path("./data/")
REPORTS_PATH = Path("./out/reports/")
SHEETS_PATH = Path("./out/sheets/")

# templates
SHEET_TPL = "sheet.html"
REPORT_TPL = "report.html"
GROUP_TPL = "group.html"

# configuration yaml validator schema
CONF_YAML_SCHEMA = {
    "titolo": { "type": "string" },
    "numero_gruppi": { "type": "integer", "min": 1, "max": 20 },
    "numero_partecipanti_per_gruppo": { "type": "integer", "min": 3, "max": 12 },
    "consegna": { "type": "string" },
    "domandaA":{ "type": "string" },
    "domandaA_scelte": { "type": "string" },
    "domandaB": { "type": "string" },
    "domandaB_scelte": { "type": "string" },
}

## 3. FUNCTIONS

### 3.2 Functions related to DOCUMENTS and DATA

In [4]:
def load_yaml_file(yaml_file, yaml_schema, validator):
    # try to load data
    try:
        # open yaml file
        with open(yaml_file, 'r') as file:
            # parse yaml data
            yaml_data = yaml.safe_load(file)
        # if yaml data validates
        if validator.validate(yaml_data, yaml_schema):
            # return yaml data and None as errors
            return (yaml_data, None)
        # on validation error
        else:
            # return None as data and errors
            return (None, validator.errors)
    # catch exceptions
    except FileNotFoundError:
        # return None as data and errors
        return(None,"Cannot locate files")
    except yaml.YAMLError as e:
        # return None as data and errors
        return (None, "Yaml files could not be parsed")
    except cerberus.DocumentError as e:
        # return None as data and errors
        return (None, "Document was loaded but cannot be evaluated")
    except cerberus.SchemaError as e:
        # return None as data and errors
        return (None, "Invalid yaml validation schema")
    
def get_sheet_data(conf_file, conf_yaml_schema):
    # init validator
    validator = Validator(required_all=True)
    # load configuration data
    yaml_data, validation_errors = load_yaml_file(DATA_PATH / conf_file, conf_yaml_schema, validator)
    # if configuration data was correctly loaded
    if yaml_data != None:
        # init sheet_data dict
        sheet_data = dict()
        # update sheet_data
        sheet_data["title"] = yaml_data["titolo"]
        sheet_data["groups"] = list(range(1, yaml_data["numero_gruppi"] +1))
        sheet_data["likert"] = string.ascii_uppercase[:yaml_data["numero_partecipanti_per_gruppo"]]
        sheet_data["explanation"] = yaml_data["consegna"]
        sheet_data["ga_question"] = yaml_data["domandaA"]
        sheet_data["ga_question_hint"] = yaml_data["domandaA_scelte"]
        sheet_data["gb_question"] = yaml_data["domandaB"]
        sheet_data["gb_question_hint"] = yaml_data["domandaB_scelte"]
        # return sheet data
        return (sheet_data, None)
    # on validation errors
    else:
        # return None and validation errors
        return (None, validation_errors)
    
def generate_doc_from_template(doc_type, doc_template, doc_data, path, prefix, suffix):
    # try to load sheet template
    try:
        # get doc template
        tpl = e.get_template(doc_template)
        # render doc
        rendered_tpl = tpl.render(doc_data);
        # build file name
        filename_html = re.sub("^_|_$", "", f"{prefix}_{doc_type}_{suffix}.html")
        filename_pdf = re.sub("^_|_$", "", f"{prefix}_{doc_type}_{suffix}.pdf")
        # save doc as pdf
        HTML(string=rendered_tpl).write_pdf(path / filename_pdf)
        # save doc as html
        # with open(path / filename_html, "w") as file: file.write(rendered_tpl)
    # catch exceptions
    except FileNotFoundError:
        return(None, f"Cannot locate {doc_type} template file")
    
def generate_yaml_group_inputs(doc_data, prefix):
    # try to load sheet template
    try:
        # get doc template
        tpl = e.get_template(GROUP_TPL)
        # loop thorugh groups
        for g in doc_data["groups"]:
            # render doc
            rendered_tpl = re.sub("^\s*\n$ ","",tpl.render(doc_data | { "groupId": g}));
            # remove blank lines
            rendered_tpl ="\n".join([ line for line in rendered_tpl.split("\n") if len(line)>0])
            # save doc as yaml
            with open(DATA_PATH / f"{prefix}_gruppo_{g}.yaml", "w") as file:
                file.write(rendered_tpl)
    # catch exceptions
    except FileNotFoundError:
        return(None, f"Cannot locate {doc_type} template file")

## 4. GENERATE

In [5]:
# init jinja environment
e = jn.Environment(loader=jn.FileSystemLoader(TEMPLATES_PATH))

In [6]:
configuration_file = "2023_rivolto.yaml"
prefix = "rivolto23"
suffix = ""

In [7]:
# notify user
print("1. Starting...")
# notify user
print(f"2. Loading file ({configuration_file})...")
# load sheet(s) data
sheet_data, sheet_errors = get_sheet_data(configuration_file, CONF_YAML_SCHEMA)
print(sheet_data)
# if sheet(s) data was correctly loaded
if (sheet_data != None):
    # notify user
    print("3. Generating doc(s)...")
    # generate sheet(s)
    generate_doc_from_template("sheet", SHEET_TPL, sheet_data, SHEETS_PATH, prefix, "")
    # generate group input doc(s)
    generate_yaml_group_inputs(sheet_data, prefix)
    # notify user
    print("4. Doc(s) generated.")
else:
    # notify user
    print(sheet_errors)
    print(report_errors)

1. Starting...
2. Loading file (2023_rivolto.yaml)...
{'title': 'Selezione Frecce - Rivolto 2023', 'groups': [1, 2], 'likert': 'ABCDEFGH', 'explanation': 'Pensa alla prova cha hai appena svolto e rispondi alle seguenti domande.', 'ga_question': 'Chi vorresti nel tuo gruppo di lavoro ideale?', 'ga_question_hint': 'Indica 2 persone, annerendo per intero la lettera corrispondente', 'gb_question': 'Chi non vorresti nel tuo gruppo di lavoro ideale?', 'gb_question_hint': 'Indica da un minimo di 1 persona a un massimo di 2 persone, annerendo per intero la lettera corrispondente'}
3. Generating doc(s)...


'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp


4. Doc(s) generated.
